In [2]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

In [3]:
# groupby 필요성 예시
data = {
    'class': ['A', 'A', 'A', 'A', 'B', 'B', 'B', 'B', 'C', 'C', 'C', 'C'],
    'sex':   ['M', 'F', 'M', 'F', 'M', 'F', 'M', 'F', 'M', 'F', 'M', 'F'],
    'score': [85, 92, 78, 95, 88, 73, 90, 82, 76, 89, 84, 91],
    'age':   [17, 18, 17, 18, 17, 18, 17, 18, 17, 18, 17, 18]
}

df = pd.DataFrame(data)

# 1. groupby 사용하지 않을 때
class_A = df[df['class'] == 'A']
class_B = df[df['class'] == 'B']
class_C = df[df['class'] == 'C']

print("A ", class_A['score'].mean())
print("B ", class_B['score'].mean())
print("C ", class_C['score'].mean())

# 2. groupby 사용할 때 (효율적)
print(df.groupby('class')['score'].mean())

A  87.5
B  83.25
C  85.0
class
A    87.50
B    83.25
C    85.00
Name: score, dtype: float64


In [4]:
# 데이터 호출
cust = pd.read_csv('../sc_cust_info_txn_v1.5.csv')
cust

,base_ym,dpro_tgt_perd_val,cust_ctg_type,cust_class,sex_type,age,efct_svc_count,dt_stop_yn,npay_yn,r3m_avg_bill_amt,r3m_A_avg_arpu_amt,r3m_B_avg_arpu_amt,r6m_A_avg_arpu_amt,r6m_B_avg_arpu_amt,termination_yn
0,202006,20200630,10001,C,F,28,0,N,N,2640.0000,792.000000,1584.0000,0.0,0.0000,Y
1,202006,20200630,10001,_,_,_,1,N,N,300.0000,90.000000,180.0000,0.0,0.0000,Y
2,202006,20200630,10001,E,F,24,1,N,N,16840.0000,2526.000000,6983.0000,0.0,6981.0000,N
3,202006,20200630,10001,F,F,32,1,N,N,15544.7334,2331.710010,6750.4666,0.0,6508.8000,N
4,202006,20200630,10001,D,M,18,1,N,N,4700.0000,0.000000,4502.0000,0.0,4507.7000,N
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9925,202006,20200630,10001,C,F,15,1,N,Y,1296.0999,194.414985,643.1001,0.0,852.5499,N
9926,202006,20200630,10001,G,M,12,1,N,N,13799.6666,2069.949990,10605.9266,0.0,10603.9266,N
9927,202006,20200630,10005,C,_,_,1,N,N,1396.2000,1206.000000,0.0000,1212.0,0.0000,N
9928,202006,20200630,10001,C,F,40,0,N,N,3140.0000,942.000000,1884.0000,0.0,0.0000,Y


In [81]:
# 그룹화 (그루핑, 전체 데이터를 조건에 맞게 변형하여 표현)

## 1. 성별 컬럼으로 그룹화 ########################################
# 그루핑하려면 컬럼이 범주형이어야 그룹을 나눌 수 있다.

# groupby
gender_group = cust.groupby('sex_type')
gender_group

## 2. groupby 내부 함수 ########################################

# groups: 그룹의 속성 확인하기
gender_group.groups

# count: 데이터 개수
gender_group.count()

# mean, var, std
# gender_group.mean() # 에러 발생, 문자열 데이터를 평균낼 수 없기 때문

# 숫자 컬럼만 나열하고 mean 구하기
cols = cust.select_dtypes(['int64', 'float64']).columns
gender_group[['r3m_avg_bill_amt', 'r3m_A_avg_arpu_amt']].mean()
gender_group[cols].mean()
gender_group[cols].var() # 분산
gender_group[cols].std() # 표준편차

# min, max
gender_group.max() # 결과가 데이터프레임 타입
gender_group.max()[['r3m_avg_bill_amt', 'r3m_A_avg_arpu_amt']] # 결과가 데이터프레임이므로, 데이터프레임에 원하는 컬럼명을 넣어 조회 가능

# size: 집단별 크기
gender_group.size()

# sum
# gender_group.sum()


## 3. 인덱스 설정(groupby) 후 데이터 추출 ########################################
# 성별 r3m_avg_bill_amt의 평균
gender_group[['r3m_avg_bill_amt']].mean()

## 4. 복수 컬럼으로 그룹화 ########################################
# groupby에 columns 리스트를 복수 개 전달할 수 있다.
class_gender_group = cust.groupby(['cust_class', 'sex_type'])
class_gender_group[['r3m_avg_bill_amt']].mean()

# multi index에서 loc 함수를 사용하여 원하는 row 선택하기
# multi_mean = class_gender_group[['r3m_avg_bill_amt']].mean()
# multi_mean.loc[[('C', 'M'), ('D', 'M')]]

## 5. index(MultiIndex)로 그룹화 ########################################
# set_index 함수 (선택한 칼럼만 분리해서 가져오기)
    # column 데이터를 index 레벨로 변경하는 경우 사용
    # 기존 인덱스를 제거하고 데이터 컬럼 중 하나를 인덱스로 설정
class_gender_index = cust.set_index(['cust_class', 'sex_type'])
class_gender_index
class_gender_index.groupby('cust_class').count()
class_gender_index.groupby('sex_type').count()
class_gender_index.groupby(['cust_class', 'sex_type']).count()

# reset_index 함수
    # 0부터 시작하는 새로운 인덱스를 생성하여 인덱스 초기화
    # 기존 인덱스는 데이터 컬럼으로 추가
# index_group.reset_index()

## 6. aggregate(집계) 함수 사용하기 ########################################
# count()를 일일히 사용하지 않고 aggregate로 한 번에 여러가지 통계 정보를 확인할 수 있다.
multi_index_group = class_gender_index.groupby(['cust_class', 'sex_type'])
# multi_index_group.aggregate(['count', 'min', 'max', 'size'])
multi_index_group[['r3m_avg_bill_amt']].aggregate(['mean', 'var', 'std', 'median', 'sum', 'prod'])
# aggregate 사용할 수 있는 값 정리
    # 공통
        # count: NaN이 아닌 값 개수
        # size: 그룹의 전채 행 수(NaN 포함)
        # first: 그룹의 첫 번째 값
        # last: 그룹의 마지막 값
    # 숫자 타입(int, float)일 경우 가능한 값
        # mean: 평균값
        # var: 분산
        # std: 표준편차
        # median: 중앙값
        # sum: 합계
        # prod: 모든 값의 곱샘


r3m_avg_bill_amt                               \
                                mean           var            std   
cust_class sex_type                                                 
C          F             3804.342244  3.260344e+07    5709.942602   
           M             3155.385796  7.148541e+06    2673.675468   
           _             4719.075980  1.733599e+07    4163.650580   
D          F             7848.842709  2.296842e+07    4792.537542   
           M             7774.098954  3.862199e+07    6214.658956   
           _             8419.764574  2.207573e+07    4698.481349   
E          F            11257.301485  2.668986e+07    5166.222695   
           M            11158.736812  3.332496e+07    5772.777275   
           _            12208.980550  7.663479e+06    2768.298850   
F          F            14913.105272  3.691025e+07    6075.380389   
           M            15013.379957  4.139155e+07    6433.626385   
           _            16231.437767  2.088105e+07    4569.578269   
G          F            16538.595017  7.710049e+07    8780.688739   
           M            16847.160637  8.629359e+07    9289.434265   
           _            20180.014811  9.909731e+07    9954.763365   
H          F            20154.938768  1.639826e+08   12805.570063   
           M            21052.154088  1.842484e+08   13573.812533   
_          F             4304.582332  1.974876e+07    4443.958100   
           M             4728.388010  8.182772e+07    9045.867490   
           _            65268.733884  4.965691e+11  704676.620691   

                                                                
                           median           sum           prod  
cust_class sex_type                                             
C          F          2545.400000  3.229887e+06            inf  
           M          2403.949950  3.357330e+06            inf  
           _          2970.800000  8.447146e+05            inf  
D          F          6789.666600  8.610180e+06            inf  
           M          6613.333400  8.450446e+06            inf  
           _          7000.566600  5.557045e+05  9.683519e+255  
E          F         10286.266600  7.193416e+06            inf  
           M         10097.555125  6.762195e+06            inf  
           _         12499.800000  2.930155e+05   6.316122e+97  
F          F         14077.366700  4.772194e+06            inf  
           M         14149.133400  5.870232e+06            inf  
           _         15826.000000  2.434716e+05   8.901084e+62  
G          F         14805.400000  1.390896e+07            inf  
           M         15408.000000  2.023344e+07            inf  
           _         19685.200000  1.816201e+05   1.908273e+38  
H          F         16540.400000  1.068212e+06  2.253281e+225  
           M         16426.387810  1.852590e+06            inf  
_          F          2954.453315  1.945671e+06            inf  
           M          2704.866600  2.652626e+06            inf  
           _          7099.500680  2.532427e+07   0.000000e+00

In [96]:
sample = pd.DataFrame(
                { '고객ID': ['cust_1', 'cust_1', 'cust_1', 'cust_2', 'cust_2', 'cust_2', 'cust_3', 'cust_3', 'cust_3'],
                  '상품코드': ['p1', 'p2', 'p3', 'p1', 'p2', 'p3', 'p1', 'p2', 'p3'],
                  '등급' : ['A', 'A', 'A', 'A', 'A', 'A', 'B', 'B', 'B'],
                  '구매금액': [30, 10, 0, 40, 15, 30, 0, 0, 10] , 
                  '나이': [15, 40, 33, 29, 6, 30, 54, 69, 38] , 
                })

sample

# pivot_table 함수 활용
    # dataframe의 형태를 변경
    # 범주형 값으로 되어있는 행 데이터를 열 데이터로 회전
    # 형태: pandas.pivot_table(data, index, columns, aggfunc)

# sample[['고객ID', '구매금액', '나이']].groupby('고객ID').mean() # 결과 동일
# pd.pivot_table(data=sample[['고객ID', '구매금액', '나이']], index='고객ID') # 결과 동일
sample[['고객ID', '구매금액', '나이']].pivot_table(index='고객ID')

sample.pivot_table(index = '등급', columns='상품코드', values='구매금액', aggfunc='sum') # 'sum' 대신 np.sum도 가능
sample.pivot_table(index = '등급', columns='상품코드', values='구매금액', aggfunc='max')
sample.pivot_table(index = '등급', columns='상품코드', values='구매금액', aggfunc='min')


상품코드,p1,p2,p3
등급,,,
A,30,10,0
B,0,0,10


In [104]:
df = pd.DataFrame(
        {
            '지역': ['서울', '서울', '서울', '경기', '경기', '부산', '서울', '서울', '부산', '경기', '경기', '경기'],
            '요일': ['월요일', '화요일', '수요일', '월요일', '화요일', '월요일', '목요일', '금요일', '화요일', '수요일', '목요일', '금요일'],
            '강수량': [100, 80, 1000, 200, 200, 100, 50, 100, 200, 100, 50, 100],
            '강수확률': [80, 70, 90, 10, 20, 30, 50, 90, 20, 80, 50, 10]
        }
    )

df

# stack, unstack 함수 활용
new_df = df.set_index(['지역', '요일'])
new_df

# stack
    # 칼럼 레벨에서 인덱스 레벨로 dataframe 변경
    # 데이터를 row 레벨로 쌓아올리는 개념
new_df.stack(0)

# unstack
    # 인덱스 레벨에서 컬럼 레벨로 dataframe 변경
# new_df.unstack('요일') # 컬럼 이름으로 접근 가능
# new_df.unstack(1) # 컬럼 인덱스로 접근 가능




지역  요일       
서울  월요일  강수량      100
         강수확률      80
    화요일  강수량       80
         강수확률      70
    수요일  강수량     1000
         강수확률      90
경기  월요일  강수량      200
         강수확률      10
    화요일  강수량      200
         강수확률      20
부산  월요일  강수량      100
         강수확률      30
서울  목요일  강수량       50
         강수확률      50
    금요일  강수량      100
         강수확률      90
부산  화요일  강수량      200
         강수확률      20
경기  수요일  강수량      100
         강수확률      80
    목요일  강수량       50
         강수확률      50
    금요일  강수량      100
         강수확률      10
dtype: int64